# 아파트 실거래 데이터 전처리
EDA 보고서(`docs/eda_report.md`) 기반 전처리 파이프라인

In [1]:
import os
import pandas as pd
import numpy as np

# 작업 디렉토리 설정 (노트북 기준 상대경로)
DATA_DIR = '../data'
os.chdir(DATA_DIR)
print(f"작업 디렉토리: {os.getcwd()}")

# 원본 데이터 로드
df = pd.read_csv('APT_Data_Prep.csv')
print(f"데이터 로드 완료: {df.shape[0]:,}행 × {df.shape[1]}열")

작업 디렉토리: /Users/taehyunan/Desktop/Repo/SeSAC/Study/sesac_ml_dl_study_repo/project/data
데이터 로드 완료: 220,890행 × 17열


In [2]:
# 전처리 전 상태 확인
print(f"원본 데이터: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"결측값 총 수: {df.isnull().sum().sum()}")
print(f"중복 행 수: {df.duplicated().sum():,}")
print(f"\n=== 컬럼 목록 ===")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}. {col} ({df[col].dtype})")

원본 데이터: 220,890행 × 17열
결측값 총 수: 0
중복 행 수: 5,746

=== 컬럼 목록 ===
   0. 거래금액 (float64)
   1. 단지명 (object)
   2. 시도명 (object)
   3. 시군구 (object)
   4. 법정동 (object)
   5. 지번 (object)
   6. 입주년도 (int64)
   7. 계약년도 (int64)
   8. 계약월 (int64)
   9. 계약일 (int64)
  10. 전용면적 (float64)
  11. 층 (int64)
  12. 경과년수 (int64)
  13. 재건축 (object)
  14. 계약일자 (object)
  15. 세대수 (int64)
  16. 주차대수 (int64)


## 1. 불필요 컬럼 제거
- **시도명**: 단일값 (서울특별시 100%) → 분석 가치 없음
- **입주년도**: 경과년수와 r=-0.99 (수학적 파생관계) → 경과년수 유지
- **주차대수**: 세대수와 r=0.95 (거의 동일 정보) → 세대수 유지

In [3]:
# 원본 보존 후 전처리 시작
df_clean = df.copy()

# 불필요 컬럼 제거
drop_cols = ['시도명', '입주년도', '주차대수']
df_clean = df_clean.drop(columns=drop_cols)
print(f"제거된 컬럼: {drop_cols}")
print(f"남은 컬럼 수: {df_clean.shape[1]}개 ({df.shape[1]} → {df_clean.shape[1]})")

제거된 컬럼: ['시도명', '입주년도', '주차대수']
남은 컬럼 수: 14개 (17 → 14)


## 2. 이상치 확인 및 처리
- **층**: min=-3 → 반지하/지하층 가능성
- **경과년수**: min=-2 → 입주 전 분양권 전매 가능성

In [4]:
# 층 음수 데이터 확인
print("=== 층 ≤ 0 데이터 ===")
floor_negative = df_clean[df_clean['층'] <= 0]
print(f"건수: {len(floor_negative):,}건 ({len(floor_negative)/len(df_clean)*100:.2f}%)")
print(floor_negative['층'].value_counts().sort_index())

print(f"\n=== 경과년수 < 0 데이터 ===")
age_negative = df_clean[df_clean['경과년수'] < 0]
print(f"건수: {len(age_negative):,}건 ({len(age_negative)/len(df_clean)*100:.2f}%)")
print(age_negative['경과년수'].value_counts().sort_index())

=== 층 ≤ 0 데이터 ===
건수: 52건 (0.02%)
층
-3     3
-2     9
-1    40
Name: count, dtype: int64

=== 경과년수 < 0 데이터 ===
건수: 29건 (0.01%)
경과년수
-2     8
-1    21
Name: count, dtype: int64


In [ ]:
# 이상치 처리
# 층 ≤ 0: 반지하/지하층 — 실제 존재하는 층이므로 유지
# 경과년수 < 0: 입주 전 분양권 전매 — 0으로 클리핑 (분양권 거래는 유효한 거래)

# 경과년수 < 0 → 0으로 클리핑
neg_age_count = (df_clean['경과년수'] < 0).sum()
df_clean['경과년수'] = df_clean['경과년수'].clip(lower=0)
print(f"경과년수 < 0 → 0 클리핑: {neg_age_count:,}건 처리")

## 3. 타입 변환
- **계약일자**: object → datetime 변환

In [6]:
# 계약일자 → datetime 변환
print(f"변환 전 dtype: {df_clean['계약일자'].dtype}")
print(f"샘플: {df_clean['계약일자'].head(3).tolist()}")

df_clean['계약일자'] = pd.to_datetime(df_clean['계약일자'])
print(f"변환 후 dtype: {df_clean['계약일자'].dtype}")

# 변환 후 개별 날짜 컬럼(계약년도, 계약월, 계약일)은 계약일자에서 추출 가능 → 제거 검토
print(f"\n계약년도/월/일 컬럼이 계약일자와 중복될 수 있음")
print(f"계약일자 범위: {df_clean['계약일자'].min()} ~ {df_clean['계약일자'].max()}")

변환 전 dtype: object
샘플: ['2020-01-31', '2020-01-06', '2020-01-14']
변환 후 dtype: datetime64[ns]

계약년도/월/일 컬럼이 계약일자와 중복될 수 있음
계약일자 범위: 2020-01-01 00:00:00 ~ 2024-12-31 00:00:00


## 4. 중복 행 제거

In [7]:
# 중복 행 제거
dup_count = df_clean.duplicated().sum()
if dup_count > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"중복 {dup_count:,}행 제거 → {df_clean.shape[0]:,}행")
else:
    print("중복 행 없음")

중복 5,746행 제거 → 215,144행


## 5. 전처리 결과 요약 및 저장

In [8]:
# 전처리 완료 요약
print("=" * 50)
print("전처리 완료 요약")
print("=" * 50)
print(f"원본 데이터:   {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"전처리 후:     {df_clean.shape[0]:,}행 × {df_clean.shape[1]}열")
print(f"제거된 행:     {df.shape[0] - df_clean.shape[0]:,}행")
print(f"제거된 컬럼:   {df.shape[1] - df_clean.shape[1]}개 ({drop_cols})")
print(f"결측값:        {df_clean.isnull().sum().sum()}")
print(f"메모리 사용:   {df_clean.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print(f"\n=== 최종 dtypes ===")
print(df_clean.dtypes)

# 전처리 데이터 저장
os.makedirs('cleaned', exist_ok=True)
df_clean.to_csv('cleaned/APT_Data_Prep_cleaned.csv', index=False)
print(f"\n저장 완료: cleaned/APT_Data_Prep_cleaned.csv")

전처리 완료 요약
원본 데이터:   220,890행 × 17열
전처리 후:     215,144행 × 14열
제거된 행:     5,746행
제거된 컬럼:   3개 (['시도명', '입주년도', '주차대수'])
결측값:        0
메모리 사용:   96.9 MB

=== 최종 dtypes ===
거래금액           float64
단지명             object
시군구             object
법정동             object
지번              object
계약년도             int64
계약월              int64
계약일              int64
전용면적           float64
층                int64
경과년수             int64
재건축             object
계약일자    datetime64[ns]
세대수              int64
dtype: object

저장 완료: cleaned/APT_Data_Prep_cleaned.csv
